In [96]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize



In [97]:
reviews_df = pd.read_csv('../Dataset/amazon_vfl_reviews.csv', encoding="UTF-8")
reviews_df = reviews_df.drop(columns=['Unnamed: 5'])

reviews_df.head()

,asin,name,date,rating,review
0,B08CN7SJBX,Maaza-1-2L,2018-11-26,2,I do have it sometimes. But the quality has be...
1,B08CN7SJBX,Maaza-1-2L,2019-11-11,4,Original and nice
2,B08CN7SJBX,Maaza-1-2L,2020-07-07,5,Best Mango Based Cool drink available in the m...
3,B08CN7SJBX,Maaza-1-2L,2020-07-02,5,Good
4,B08CN7SJBX,Maaza-1-2L,2020-04-30,1,When the product arrive it was kind of dirty. ...


In [98]:
reviews_df.shape

(895, 5)

In [99]:
reviews_df.isnull().sum()


asin      0
name      0
date      0
rating    0
review    2
dtype: int64

In [100]:
# the review column, four rows without review text, we drop the rows with the null columns
reviews_df = reviews_df.dropna()
#resetting the index
reviews_df = reviews_df.reset_index(drop=True)

In [101]:
# remove all characters not number or characters
def cleanText(input_string):
    modified_string = re.sub('[^A-Za-z0-9]+', ' ', input_string)
    return(modified_string)
reviews_df['review'] = reviews_df.review.apply(cleanText)


In [102]:
# From the name we extract the brand
reviews_df['brandName'] = reviews_df['name'].str.split('-').str[0]
reviews_df.head()

,asin,name,date,rating,review,brandName
0,B08CN7SJBX,Maaza-1-2L,2018-11-26,2,I do have it sometimes But the quality has bee...,Maaza
1,B08CN7SJBX,Maaza-1-2L,2019-11-11,4,Original and nice,Maaza
2,B08CN7SJBX,Maaza-1-2L,2020-07-07,5,Best Mango Based Cool drink available in the m...,Maaza
3,B08CN7SJBX,Maaza-1-2L,2020-07-02,5,Good,Maaza
4,B08CN7SJBX,Maaza-1-2L,2020-04-30,1,When the product arrive it was kind of dirty T...,Maaza


In [103]:
reviews_df['brandName'].value_counts()


brandName
Society        182
Tata           180
Amul           164
Britannia      119
Patanjali       70
PaperBoat       40
Natural         40
Maaza           20
CocaCola        20
Maggi           20
GluconD         20
NutriChoice     12
Indiana          6
Name: count, dtype: int64

In [104]:
reviews_df['brandName'] = reviews_df['brandName'].str.title()
reviews_df.brandName.unique()

array(['Maaza', 'Paperboat', 'Indiana', 'Cocacola', 'Natural', 'Maggi',
       'Glucond', 'Amul', 'Patanjali', 'Britannia', 'Nutrichoice',
       'Society', 'Tata'], dtype=object)

In [105]:
# Extracting the product from the name column
products = []
for value in reviews_df['name']:
    indx = len(value.split('-')[0])+1
    products.append(value[indx:])
reviews_df['product'] = products
reviews_df['product'].unique()

array(['1-2L', 'Aamras-Juice-250ml', 'Frutti-Cherries-Frooti-Multicolor',
       '300ml-Pack-6', 'Mango-Juice-1L-Pack', 'Mixed-Fruit',
       'Masala-Noodles-Singles-840g', 'Glucose-Based-Beverage-Orange',
       'Festive-Delight-Assorted-Jelimals',
       'Butter-Pasteurised-100g-Pack', 'Cow-Ghee-500ml',
       'Fresh-Paneer-200g', 'Cacao-Chocolate-125g-Pack',
       'Cheese-Slices-200g-Pack', 'Fresh-Cream-250ml',
       'Pure-Ghee-Pouch-1L', 'Pro-500g-Pouch-Pack',
       'Gold-Milk-Homogenised-Standardised', 'Cows-Ghee-1L',
       'Shatavar-Mushli-Ashwagandha-Churna', 'UHT-Milk-1000-ml',
       'Cows-Ghee-500ml', 'NutriChoice-Digestive-100g',
       'Good-Day-Cashew-200g', 'Marie-Gold-200g',
       'Digestives-grain-Pack-200g', 'Vita-Marie-Gold-150g',
       '50-50-Biscuits-Pouch-80g', 'Nice-Time-150g',
       'Tea-Regular-250g-Pouch', 'Tea-Regular-Pouch-1kg',
       'Tea-Masala-Jar-250g', 'Tea-Minute-Masala-500g',
       'Daily-Elachi-Premix-Pouch', 'Tea-Premium-Packet-1000g',
     

In [106]:
reviews_df.head()


,asin,name,date,rating,review,brandName,product
0,B08CN7SJBX,Maaza-1-2L,2018-11-26,2,I do have it sometimes But the quality has bee...,Maaza,1-2L
1,B08CN7SJBX,Maaza-1-2L,2019-11-11,4,Original and nice,Maaza,1-2L
2,B08CN7SJBX,Maaza-1-2L,2020-07-07,5,Best Mango Based Cool drink available in the m...,Maaza,1-2L
3,B08CN7SJBX,Maaza-1-2L,2020-07-02,5,Good,Maaza,1-2L
4,B08CN7SJBX,Maaza-1-2L,2020-04-30,1,When the product arrive it was kind of dirty T...,Maaza,1-2L


In [107]:
reviews_df['clean_review_text']=reviews_df['review'].str.lower()


In [108]:
#removing punctuations
reviews_df['clean_review_text']=reviews_df['clean_review_text'].str.translate(str.maketrans('','',string.punctuation))

In [109]:
stopWords=stopwords.words('english')+['the', 'a', 'an', 'i', 'he', 'she', 'they', 'to', 'of', 'it', 'from']
def removeStopWords(stopWords, rvw_txt):
    newtxt = ' '.join([word for word in rvw_txt.split() if word not in stopWords])
    return newtxt
reviews_df['clean_review_text'] = [removeStopWords(stopWords,x) for x in reviews_df['clean_review_text']]

In [110]:
nltk.download('vader_lexicon')
sentiment_model = SentimentIntensityAnalyzer()
sentiment_scores=[]
sentiment_score_flag = []
for text in reviews_df['clean_review_text']:
        sentimentResults = sentiment_model.polarity_scores(text)
        sentiment_score = sentimentResults["compound"]
        #print(sentimentResults)
        #The compound value reflects the overall sentiment ranging from -1 being very negative and +1 being very positive.
        sentiment_scores.append(sentiment_score)
        # marking the sentiments as positive, negative and neutral 
        if sentimentResults['compound'] >= 0.05 : 
            sentiment_score_flag.append('positive')
  
        elif sentimentResults['compound'] <= - 0.05 : 
            sentiment_score_flag.append('negative')
  
        else : 
            sentiment_score_flag.append('neutral')
            
reviews_df['scores']=sentiment_scores
reviews_df['scoreStatus'] = sentiment_score_flag


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/adi/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [111]:
reviews_df


,asin,name,date,rating,review,brandName,product,clean_review_text,scores,scoreStatus
0,B08CN7SJBX,Maaza-1-2L,2018-11-26,2,I do have it sometimes But the quality has bee...,Maaza,1-2L,sometimes quality reduced,0.0000,neutral
1,B08CN7SJBX,Maaza-1-2L,2019-11-11,4,Original and nice,Maaza,1-2L,original nice,0.6249,positive
2,B08CN7SJBX,Maaza-1-2L,2020-07-07,5,Best Mango Based Cool drink available in the m...,Maaza,1-2L,best mango based cool drink available market p...,0.7184,positive
3,B08CN7SJBX,Maaza-1-2L,2020-07-02,5,Good,Maaza,1-2L,good,0.4404,positive
4,B08CN7SJBX,Maaza-1-2L,2020-04-30,1,When the product arrive it was kind of dirty T...,Maaza,1-2L,product arrive kind dirty product expiry date ...,-0.4588,negative
...,...,...,...,...,...,...,...,...,...,...
888,B079H8Y973,Tata-Agni-Leaf-Tea-1kg,2019-10-09,3,Tata Agni Tea is value for money It s MRP is 2...,Tata,Agni-Leaf-Tea-1kg,tata agni tea value money mrp 220 overall good...,0.7783,positive
889,B079H8Y973,Tata-Agni-Leaf-Tea-1kg,2020-09-04,5,200It s a truly good deal for you Market pric...,Tata,Agni-Leaf-Tea-1kg,200it truly good deal market price per 100 gra...,0.9521,positive
890,B079H8Y973,Tata-Agni-Leaf-Tea-1kg,2019-11-13,1,Expected better quality tea from a brand like ...,Tata,Agni-Leaf-Tea-1kg,expected better quality tea brand like tata te...,0.3818,positive
891,B079H8Y973,Tata-Agni-Leaf-Tea-1kg,2019-02-17,5,Tata Tea Agni is good if you re looking for go...,Tata,Agni-Leaf-Tea-1kg,tata tea agni good looking good less priced op...,0.9413,positive


In [112]:
# Calculate sentiment percentages grouped by brand and product
sentiment_analysis = reviews_df.groupby(['brandName', 'product', 'scoreStatus']).size().unstack(fill_value=0)
sentiment_pct = sentiment_analysis.div(sentiment_analysis.sum(axis=1), axis=0) * 100

# Add total reviews count
total_reviews = reviews_df.groupby(['brandName', 'product']).size()

# Print results for each brand and product
for brand in reviews_df['brandName'].unique():
    print(f"\nBrand: {brand}")
    products = reviews_df[reviews_df['brandName'] == brand]['product'].unique()
    
    for product in products:
        print(f"\nProduct: {product}")
        if (brand, product) in sentiment_pct.index:
            print("Sentiment Distribution:")
            print(f"Positive: {sentiment_pct.loc[(brand, product), 'positive']:.1f}%")
            print(f"Neutral: {sentiment_pct.loc[(brand, product), 'neutral']:.1f}%")
            print(f"Negative: {sentiment_pct.loc[(brand, product), 'negative']:.1f}%")
            print(f"Total Reviews: {total_reviews.loc[(brand, product)]}")


Brand: Maaza

Product: 1-2L
Sentiment Distribution:
Positive: 60.0%
Neutral: 30.0%
Negative: 10.0%
Total Reviews: 20

Brand: Paperboat

Product: Aamras-Juice-250ml
Sentiment Distribution:
Positive: 90.0%
Neutral: 0.0%
Negative: 10.0%
Total Reviews: 20

Product: Mixed-Fruit
Sentiment Distribution:
Positive: 80.0%
Neutral: 10.0%
Negative: 10.0%
Total Reviews: 20

Brand: Indiana

Product: Frutti-Cherries-Frooti-Multicolor
Sentiment Distribution:
Positive: 100.0%
Neutral: 0.0%
Negative: 0.0%
Total Reviews: 6

Brand: Cocacola

Product: 300ml-Pack-6
Sentiment Distribution:
Positive: 80.0%
Neutral: 0.0%
Negative: 20.0%
Total Reviews: 20

Brand: Natural

Product: Mango-Juice-1L-Pack
Sentiment Distribution:
Positive: 70.0%
Neutral: 0.0%
Negative: 30.0%
Total Reviews: 20

Product: Festive-Delight-Assorted-Jelimals
Sentiment Distribution:
Positive: 80.0%
Neutral: 20.0%
Negative: 0.0%
Total Reviews: 20

Brand: Maggi

Product: Masala-Noodles-Singles-840g
Sentiment Distribution:
Positive: 50.0%
Neu